# SparseWalker Experiment 34 — backward-free graph learning

This is the first deliberately narrow **no-backward** SparseWalker experiment. It does **not** claim to learn the whole representation from scratch without gradients. Instead it asks the most natural first question:

> Can SparseWalker's sparse graph dynamics be relearned locally, with no `backward()`, once the item/router/concept representation exists?

The experiment loads the best corrected Beauty SparseWalker v1.1 checkpoint from Experiment 33, freezes every parameter, resets `graph.edge_logits` to zero, and relearns only those four outgoing edge biases using the local rule

`source_mass × p(edge|context) × (reward(edge) - expected_reward)`

where reward is next-item compatibility of the destination concept. The learning loop is under `torch.no_grad()`, has no optimizer, and asserts that no parameter gradient tensors are created.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, json, runpy, torch
from pathlib import Path

REPO='/content/Sparsewalker'
BRANCH='agent/backward-free-walker'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO], check=True)
sys.path.insert(0, f'{REPO}/src')
sys.path.insert(0, f'{REPO}/experiments')
import sparsewalker
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU', torch.cuda.get_device_name(0))
print('torch', torch.__version__, 'bf16', torch.cuda.is_bf16_supported())
print('BRANCH', BRANCH)
print('sparsewalker', sparsewalker.__file__)


## Run Beauty backward-free graph relearning

The default uses the Beauty checkpoint already produced by notebook 33 at `MyDrive/sparsewalker_amazon_quality_v11/beauty/seed42/SparseWalker/best.pt`. It first prints the original pretrained quality, then quality after zeroing graph edge logits, then 12 local-learning epochs.


In [ ]:
SCRIPT=f'{REPO}/experiments/run_amazon_backward_free_graph.py'
sys.argv=[
    SCRIPT,
    '--dataset','beauty',
    '--epochs','12',
    '--batch-size','512',
    '--eval-batch-size','1024',
    '--local-lr','0.5',
    '--reset-edge-logits',
]
runpy.run_path(SCRIPT, run_name='__main__')


## Read the result

The most useful number is `recovery_fraction_val`:

- `0`: local learning did not recover anything lost by resetting the learned edge biases.
- `1`: it recovered the entire validation-quality loss without backpropagation.
- `>1`: the local rule improved beyond the original learned edge biases.

Also compare the final test NDCG@10 to the original Beauty SparseWalker reference (~0.04488 from Experiment 33).


In [ ]:
p=Path('/content/drive/MyDrive/sparsewalker_backward_free_graph/beauty/seed42/result.json')
r=json.loads(p.read_text())
print(json.dumps(r, indent=2))

pre=r['pretrained']['test']['NDCG@10']
reset=r['after_edge_reset']['test']['NDCG@10']
bf=r['best_backward_free']['test']['NDCG@10']
print('TEST NDCG@10', {'pretrained':pre, 'edge_reset':reset, 'backward_free':bf, 'bf_vs_pretrained':bf-pre})


## Interpretation

This is a **graph-plasticity** experiment, not yet a completely gradient-free model. The item embeddings, router, concept keys/values and context projection were originally learned by the standard Walker checkpoint and are frozen here.

If local graph relearning recovers a substantial fraction of the reset quality, the next backward-free step is to replace gradient learning of the router/concept geometry. If it fails, we should inspect whether static edge biases matter at all and then test local structural rewiring rather than forcing this rule.
